<a href="https://colab.research.google.com/github/hwangho-kim/Transformer_Fewshot_PdM/blob/main/CSV_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EC%83%9D%EC%84%B1%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import os

# 데이터 저장 폴더
output_dir = "factory_data"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print("Starting to generate raw factory FDC data...")

NUM_WAFERS = 1000
NUM_SENSORS = 7
sensor_names = ['RF Power (W)', 'Pressure (mTorr)', 'ESC Temp (C)', 'C4F8 Gas (sccm)', 'O2 Gas (sccm)', 'Ar Gas (sccm)', 'Vpp (V)']

np.random.seed(42)

trace_records = []
y_data = []

for w_idx in range(NUM_WAFERS):
    latent_drift = np.random.randn() * 0.5

    # 실제 장비처럼 웨이퍼마다 공정 소요 시간이 다름 (40 ~ 60 스텝)
    random_length = np.random.randint(40, 61)
    trace = np.zeros((random_length, NUM_SENSORS))

    step1_end = int(random_length * 0.1)
    step2_end = int(random_length * 0.8)

    # 정상 파형 생성
    trace[step1_end:step2_end, 0] = 1000 - latent_drift * 10
    trace[:step1_end, 1] = np.linspace(10, 30, step1_end)
    trace[step1_end:step2_end, 1] = 30 + latent_drift * 1.5
    trace[step2_end:, 1] = np.linspace(30, 10, random_length - step2_end)
    trace[:, 2] = 60 + latent_drift * 2
    trace[:step1_end, 3] = np.linspace(0, 40, step1_end)
    trace[step1_end:step2_end, 3] = 40
    trace[:step1_end, 4] = np.linspace(0, 15, step1_end)
    trace[step1_end:step2_end, 4] = 15
    trace[:, 5] = 200
    trace[step1_end:step2_end, 6] = 800 + latent_drift * 25

    # [이상 시나리오 주입] - 마지막 3장의 웨이퍼
    if w_idx == NUM_WAFERS - 3:
        # Scenario A: C4F8 Gas Hunting
        hunting_wave = 20 * np.sin(np.linspace(0, 15 * np.pi, step2_end - step1_end))
        trace[step1_end:step2_end, 3] += hunting_wave
    elif w_idx == NUM_WAFERS - 2:
        # Scenario B: Pressure Sluggish Ramp-up
        slow_ramp_end = int(random_length * 0.6)
        trace[0:slow_ramp_end, 1] = np.linspace(10, 30, slow_ramp_end)
    elif w_idx == NUM_WAFERS - 1:
        # Scenario C: RF Power Sudden Glitch
        glitch_idx = int(random_length * 0.5)
        trace[glitch_idx:min(glitch_idx+3, random_length), 0] = 0
        trace[glitch_idx:min(glitch_idx+3, random_length), 6] = 0

    noise_scales = [5.0, 0.5, 0.2, 0.5, 0.2, 2.0, 8.0]
    noise = np.random.randn(random_length, NUM_SENSORS) * noise_scales
    final_trace = trace + noise

    # DB 포맷(Long format)으로 레코드 저장
    for t_idx in range(random_length):
        record = [w_idx, t_idx] + final_trace[t_idx, :].tolist()
        trace_records.append(record)

    # 계측 데이터 생성
    etch_depth = 150 - latent_drift * 4.5 + np.random.randn() * 0.8
    if w_idx >= NUM_WAFERS - 3:
        etch_depth -= np.random.randint(10, 30)
    y_data.append(etch_depth)

# DataFrame 변환 및 CSV 저장
df_trace = pd.DataFrame(trace_records, columns=['Wafer_ID', 'Time_Step'] + sensor_names)
trace_csv_path = os.path.join(output_dir, 'raw_fdc_trace.csv')
df_trace.to_csv(trace_csv_path, index=False)

df_metrology = pd.DataFrame({'Wafer_ID': range(NUM_WAFERS), 'Etch_Depth_nm': y_data})
metro_csv_path = os.path.join(output_dir, 'raw_metrology.csv')
df_metrology.to_csv(metro_csv_path, index=False)

print(f"Data generation complete! Saved to:\n- {trace_csv_path}\n- {metro_csv_path}")